# How to work with the database
The functions can be found in db.py. The database is built using [FAISS](https://github.com/facebookresearch/faiss) and [Redis](https://redis.io/)

In [4]:
import tqdm
from wp1.database.db import *
from wp1.models.model import *
from pathlib import Path
from PIL import Image
from wp1.dataset.dataset import open_and_convert_image

### Step 1: Device selection

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Step 2: Model creation
Place your pretrained weights in the weight_fold prior to starting the command, and replace the model name by the name of the model you plan in using - see *model_usage.ipynb* for more information

In [6]:
weight_path = Path("../../../../../weight_fold/resnet/model/last_epoch_new")
model = Model(model_name="resnet", weight=weight_path, device=device)

### Step 3: Building the database

In [7]:
# Creating a new database
db_name = "new_db" 
db = Database(db_name, model, load=False, device=device)

Creating a new FAISS index
Transferring index to GPU
Index transferred to GPU


In [8]:
load = True
if load:
    # Loading an existing database
    db_name = "new_db"
    db = Database(db_name, model, load=True, device=device)
    print(f"The database contains {db.__len__()} elements.")
else:
    # Add images
    dataset_path = Path("../../../../../Data/patches/CrC/NCT-CRC-HE-100K/")
    db.add_dataset(dataset_path)
    print(f"The database contains {db.__len__()} elements.")

Loaded 200001 keys into Redis.
Transferring index to GPU
Index transferred to GPU
The database contains 100000 elements.


### Step 4: Training the database (optional)

In [9]:
db.train_index()

AttributeError: 'GpuIndexIVFFlat' object has no attribute 'index'

### Step 5: Save the database

In [10]:
db.save()

Saving Redis data to new_db.json
Saved FAISS index and 200001 Redis keys to disk.


### Step 6: Search in the database

In [ ]:
nb_neighbors = 10
query = Path("../../../../../Data/patches/CrC/CRC-VAL-HE-7K/ADI/ADI-TCGA-AAICEQFN.tif")
query_im = open_and_convert_image(query)
values, distances, t_model, t_search, t_transfer = db.search(query_im, nrt_neigh = nb_neighbors)

print("The ", nb_neighbors, " the closest to the query are: ", values,". They are at distances: ", distances)
print("The model took ", t_model, " to process the query, while the search in the database took ", t_search)

NameError: name 'open_and_convert' is not defined